In [5]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline



In [6]:
words = open("names.txt", "r").read().splitlines()

In [7]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [102]:
def build_dataset(words):

    block_size = 5
    X, Y = [], []

    for w in words:

        context = [0] * block_size

        for ch in w + ".":
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]

    X = torch.tensor(X)
    Y = torch.tensor(Y)

    return X, Y

import random

random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,  Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

In [103]:
C = torch.randn((27, 10))
w1 = torch.randn((50, 200))
b1 = torch.randn(200)
w2 = torch.randn((200,27))
b2 = torch.randn(27)
parameters = [C, w1, b1, w2, b2]

for p in parameters:
    p.requires_grad = True

In [104]:
lre = torch.linspace(-3, 0, 1000)
lrs = 10 ** lre

In [105]:
lossi = []
lri = []
stepi = []

In [106]:
for i in range (200000):

    # Minibatch Construct:
    ix = torch.randint(0, Xtr.shape[0], (32, )) #type:ignore

    # Forward:
    emb = C[Xtr[ix]] # (32, 3, 2)
    h = torch.tanh(emb.view(-1, 50) @ w1 + b1) # (32, 100)
    logits = h @ w2 + b2 # (32, 27)

    # ==========
    #counts = logits.exp()
    #prob = counts / counts.sum(1, keepdim=True)
    #loss = -prob[torch.arange(32), Y].log().mean()

    # All the above steps are less efficient yet equal to:
    loss = F.cross_entropy(logits, Ytr[ix]) #type:ignore
    # Backward:
    for p in parameters:
        p.grad = None
    loss.backward()

    # Update
    lr = 0.1 if i < 100000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad #type: Ignore

    # Track
    # lri.append(lre[i])
    lossi.append(loss.log10().item())
    stepi.append(i)

print(loss)

tensor(2.1832, grad_fn=<NllLossBackward0>)


In [107]:
emb = C[Xtr]
h = torch.tanh(emb.view(-1, 50) @ w1 + b1)
logits = h @ w2 + b2
loss = F.cross_entropy(logits, Ytr)
loss

tensor(2.1718, grad_fn=<NllLossBackward0>)

In [108]:
emb = C[Xdev]
h = torch.tanh(emb.view(-1, 50) @ w1 + b1)
logits = h @ w2 + b2
loss = F.cross_entropy(logits, Ydev)
loss

tensor(2.1916, grad_fn=<NllLossBackward0>)